# Cálculo de Anomalías Gravimétricas Isostáticas

Este notebook calcula las **anomalías isostáticas de gravedad (AIG)** a partir de rasters de topografía (DEM) y gravedad observada.

El proceso incluye:
1. **Matriz auxiliar de compartimentos**: Promedio de elevaciones en anillos concéntricos alrededor de cada celda.
2. **Reducción de Aire Libre (RAL)**: Corrección por elevación sobre el geoide.
3. **Reducción por Placa de Bouguer (RPB)**: Corrección por la masa entre la estación y el geoide.
4. **Reducción por Topografía (RTo)**: Corrección por irregularidades del terreno circundante.
5. **Reducción por Isostasia (RIs)**: Corrección por compensación isostática (modelo Airy-Heiskanen).
6. **Anomalía Isostática**: AIG = Gravedad observada + RPB + RTo + RIs.
7. **Mosaico**: Fusión de los rasters resultantes en un único raster.

In [ ]:
import os
import glob
from math import sqrt, atan, pi

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.merge import merge
from rasterio.warp import reproject, calculate_default_transform
from rasterio.io import MemoryFile
from rasterio.windows import Window

## Configuración de rutas

- `path_rasterDEM`: Directorio con los rasters de topografía (DEM) particionados.
- `path_gravedad`: Directorio con los rasters de gravedad observada.
- `path_isostasia_output`: Directorio de salida para los rasters de anomalía isostática.

In [ ]:
path_rasterDEM = "/home/jasonromeroia/Documents/Personal/TesisUDFJCMCIC/solucion2025/data/raster/topografiaZona"
path_gravedad = "/home/jasonromeroia/Documents/Personal/TesisUDFJCMCIC/solucion2025/data/raster/gravedadZona"
path_isostasia_output = "/home/jasonromeroia/Documents/Personal/TesisUDFJCMCIC/solucion2025/data/raster/isostasia/"

## Función principal: Cálculo de anomalías gravimétricas

La función `anomaly_gravimetry` procesa cada raster DEM y su correspondiente raster de gravedad para calcular la anomalía isostática.

**Parámetros de compartimentos (Hammer zones):**
- Se definen 11 zonas anulares con radios desde 590 m hasta 166,700 m.
- Cada zona se subdivide en sectores angulares (8 a 28 según la zona).
- Total: 184 compartimentos + 1 para la topografía central = 185.

**Constantes físicas:**
- G = 6.6742 × 10⁻¹¹ m³/(kg·s²) (constante gravitacional)
- ρ = 2670 kg/m³ (densidad de la corteza)
- t = 30,000 m (profundidad de compensación isostática)

In [ ]:
def anomaly_gravimetry(path_dem, path_gravedad, path_output, margin_pix=390):
    """Calcula anomalías isostáticas de gravedad para cada raster DEM.

    Parameters
    ----------
    path_dem : str
        Directorio con rasters de topografía (.tif).
    path_gravedad : str
        Directorio con rasters de gravedad observada (.tif).
    path_output : str
        Directorio de salida para los rasters de anomalía isostática.
    margin_pix : int
        Margen en píxeles para evitar efectos de borde (default: 390).
    """
    # Constantes físicas
    G = 6.6742e-11       # Constante gravitacional (m³/(kg·s²))
    rho = 2670.0         # Densidad corteza (kg/m³)
    t = 30000.0          # Profundidad de compensación isostática (m)

    # Definición de compartimentos (Hammer zones)
    aux1 = [1.0, 9.0, 19.0, 31.0, 47.0, 67.0, 83.0, 103.0, 127.0, 141.0, 157.0]
    aux2 = [8.0, 10.0, 12.0, 16.0, 20.0, 16.0, 20.0, 24.0, 14.0, 16.0, 28.0]
    aux_dist = [590.0, 1280.0, 2290.0, 3520.0, 5240.0, 8440.0, 12400.0,
                18800.0, 28800.0, 58800.0, 99000.0, 166700.0]

    list_raster = [os.path.join(path_dem, f)
                   for f in os.listdir(path_dem) if f.endswith('.tif')]

    for raster_dem in list_raster:
        name_gravedad = raster_dem.split("/")[-1].replace("topografiaZona", "")
        raster_gravedad = os.path.join(path_gravedad, name_gravedad)
        print(f"DEM: {raster_dem}")
        print(f"Gravedad: {raster_gravedad}")

        # --- Leer DEM ---
        with rasterio.open(raster_dem) as src_dem:
            topo = src_dem.read(1)
            dem_transform = src_dem.transform
            dem_crs = src_dem.crs

        filas, columnas = topo.shape

        # =================================================================
        # PASO 1: Matriz auxiliar de compartimentos
        # Promedio de elevaciones en anillos concéntricos/sectores
        # =================================================================
        correc = np.zeros((filas, columnas, 185))
        correc[:, :, 0] = topo

        for i in range(margin_pix, filas - margin_pix):
            for j in range(margin_pix, columnas - margin_pix):
                aux_cont = np.zeros(185)
                aux_n = np.zeros(185)

                for k in range(i - 167, i + 1):
                    aux_for1 = int(sqrt(27889.0 - (k - i) ** 2) + j)

                    for m in range(j, aux_for1 + 1):
                        dist = sqrt((i * 500.0 - k * 500.0) ** 2 +
                                    (j * 500.0 - m * 500.0) ** 2)
                        azim = 90.0 - atan((i - k) / (m - j)) * 180.0 / pi

                        for q in range(11):
                            if aux_dist[q] <= dist <= aux_dist[q + 1]:
                                n_sectors = int(np.ceil(aux2[q] / 4))
                                for ang in range(1, n_sectors + 1):
                                    if ((ang - 1.0) * 360.0 / aux2[q]) <= azim <= (ang * 360.0 / aux2[q]):
                                        # Noreste
                                        idx = int(aux1[q] + ang)
                                        aux_cont[idx] += topo[k, m]
                                        aux_n[idx] += 1.0
                                        # Noroeste
                                        idx = int(aux1[q] + aux2[q] - ang + 1)
                                        aux_cont[idx] += topo[k, 2 * j - m]
                                        aux_n[idx] += 1.0
                                        # Sureste
                                        idx = int(aux1[q] + (aux2[q] / 2) - ang + 1)
                                        aux_cont[idx] += topo[2 * i - k, m]
                                        aux_n[idx] += 1.0
                                        # Suroeste
                                        idx = int(aux1[q] + (aux2[q] / 2) + ang)
                                        aux_cont[idx] += topo[2 * i - k, 2 * j - m]
                                        aux_n[idx] += 1.0

                for comp in range(1, 185):
                    if aux_n[comp] > 0:
                        correc[i, j, comp] = aux_cont[comp] / aux_n[comp]

        print("Paso 1: Matriz auxiliar de compartimentos completada")

        # =================================================================
        # PASO 2: Reducción de Aire Libre (RAL)
        # =================================================================
        RAL = np.zeros(topo.shape)
        RAL[topo > 0] = 0.3086 * topo[topo > 0]
        print("Paso 2: Reducción de Aire Libre completada")

        # =================================================================
        # PASO 3: Reducción por Placa de Bouguer (RPB)
        # =================================================================
        RPB = np.zeros(topo.shape)
        RPB[topo > 0] = -0.1119 * topo[topo > 0]
        RPB[topo <= 0] = -0.0690 * topo[topo <= 0]
        print("Paso 3: Reducción por Placa de Bouguer completada")

        # =================================================================
        # PASO 4: Reducciones por Topografía (RTo) e Isostasia (RIs)
        # =================================================================
        RTo = np.zeros(topo.shape)
        RIs = np.zeros(topo.shape)

        for i in range(margin_pix, filas - margin_pix):
            for j in range(margin_pix, columnas - margin_pix):
                delta_sum_t = 0.0
                delta_sum_i = 0.0

                # Primer compartimento (a1=0, a2=aux_dist[0])
                a1 = 0
                a2 = aux_dist[0]

                if topo[i, j] > 0:
                    delta_sum_t = a2

                if topo[i, j] >= 0:
                    bI = 4.45 * topo[i, j]
                    cI = t + 5.45 * topo[i, j]
                else:
                    bI = -2.73 * topo[i, j]
                    cI = t
                    delta_sum_i = -(sqrt(a2**2 + (cI - bI)**2) -
                                    sqrt(a1**2 + (cI - bI)**2) -
                                    sqrt(a2**2 + cI**2) +
                                    sqrt(a1**2 + cI**2))
                    bI = -topo[i, j]
                    cI = bI

                delta_sum_i += (sqrt(a2**2 + (cI - bI)**2) -
                                sqrt(a1**2 + (cI - bI)**2) -
                                sqrt(a2**2 + cI**2) +
                                sqrt(a1**2 + cI**2))

                # Demás compartimentos (W = 1..184)
                E = 0
                for W in range(1, 185):
                    if W > aux1[E] + aux2[E]:
                        E += 1
                    a1 = aux_dist[E]
                    a2 = aux_dist[E + 1]
                    n = aux2[E]

                    # Corrección por topografía
                    if topo[i, j] > 0:
                        if topo[i, j] >= correc[i, j, W]:
                            bT = correc[i, j, W] - topo[i, j]
                            cT = 0
                        else:
                            bT = topo[i, j] - correc[i, j, W]
                            cT = bT

                        delta_sum_t += ((1 / n) *
                                        (sqrt(a2**2 + (cT - bT)**2) -
                                         sqrt(a1**2 + (cT - bT)**2) -
                                         sqrt(a2**2 + cT**2) +
                                         sqrt(a1**2 + cT**2)))

                    # Corrección por isostasia
                    if correc[i, j, W] >= 0:
                        bI = 4.45 * correc[i, j, W]
                        cI = correc[i, j, W] + t + bI
                    else:
                        bI = -2.73 * correc[i, j, W]
                        cI = t
                        delta_sum_i += ((1 / n) *
                                        (sqrt(a2**2 + (cI - bI)**2) -
                                         sqrt(a1**2 + (cI - bI)**2) -
                                         sqrt(a2**2 + cI**2) +
                                         sqrt(a1**2 + cI**2)))
                        bI = -correc[i, j, W]
                        cI = bI

                    delta_sum_i += ((1 / n) *
                                    (sqrt(a2**2 + (cI - bI)**2) -
                                     sqrt(a1**2 + (cI - bI)**2) -
                                     sqrt(a2**2 + cI**2) +
                                     sqrt(a1**2 + cI**2)))

                RTo[i, j] = 2.0 * pi * G * rho * delta_sum_t
                RIs[i, j] = 2.0 * pi * G * rho * delta_sum_i

        print("Paso 4: Reducciones por Topografía e Isostasia completadas")

        # =================================================================
        # PASO 5: Leer gravedad y reproyectar si es necesario
        # =================================================================
        with rasterio.open(raster_dem) as src_topo:
            topo = src_topo.read(1)
            topo_transform = src_topo.transform
            topo_crs = src_topo.crs
            topo_shape = topo.shape

        with rasterio.open(raster_gravedad) as src_grav:
            grav = src_grav.read(1)
            grav_transform = src_grav.transform
            grav_crs = src_grav.crs

            if topo_crs != grav_crs:
                print("Reproyectando gravedad al CRS de la topografía...")
                transform_repr, width, height = calculate_default_transform(
                    grav_crs, topo_crs, src_grav.width, src_grav.height,
                    *src_grav.bounds)
                grav_reprojected = np.empty((height, width), dtype=grav.dtype)
                reproject(
                    source=grav,
                    destination=grav_reprojected,
                    src_transform=grav_transform,
                    dst_transform=transform_repr,
                    src_crs=grav_crs,
                    dst_crs=topo_crs,
                    resampling=Resampling.bilinear
                )
                src_transform_for_resize = transform_repr
            else:
                print("Las proyecciones coinciden.")
                grav_reprojected = grav
                src_transform_for_resize = grav_transform

        # Reajustar gravedad al tamaño de la topografía
        grav_resized = np.empty(topo_shape, dtype=grav.dtype)
        reproject(
            source=grav_reprojected,
            destination=grav_resized,
            src_transform=src_transform_for_resize,
            dst_transform=topo_transform,
            src_crs=topo_crs,
            dst_crs=topo_crs,
            resampling=Resampling.bilinear
        )
        print("Paso 5: Gravedad reproyectada y redimensionada")

        # =================================================================
        # PASO 6: Anomalía Isostática de Gravedad (AIG)
        # La RAL ya está incluida en los datos de gravedad observada
        # =================================================================
        AIG = grav_resized + RPB + RTo + RIs
        print(f"Paso 6: AIG calculada (shape: {AIG.shape})")

        # =================================================================
        # PASO 7: Guardar resultado
        # =================================================================
        name_output = raster_dem.split("/")[-1]
        raster_output = os.path.join(path_output, name_output)

        with rasterio.open(
            raster_output, 'w', driver='GTiff',
            height=AIG.shape[0], width=AIG.shape[1],
            count=1, dtype=AIG.dtype,
            crs=topo_crs, transform=topo_transform
        ) as dst:
            dst.write(AIG, 1)

        print(f"Guardado: {raster_output}\n")

## Ejecución

Se procesan los 4 rasters particionados de Colombia con un margen de 390 píxeles (195 km a cada lado).

In [ ]:
anomaly_gravimetry(path_rasterDEM, path_gravedad, path_isostasia_output, margin_pix=390)

## Mosaico de anomalías isostáticas

Se fusionan los 4 rasters de anomalía isostática en un único mosaico, recortando 1 píxel de borde para evitar artefactos en las uniones.

In [ ]:
isostasia_folder = path_isostasia_output
isostasia_files = [f for f in glob.glob(os.path.join(isostasia_folder, "*.tif"))
                   if "mosaico" not in os.path.basename(f)]

border = 1  # píxeles a recortar en cada lado

memfiles = []
src_files_to_mosaic = []

for f in isostasia_files:
    with rasterio.open(f) as src:
        print(f"Abriendo {os.path.basename(f)} (nodata: {src.nodata})")
        win = Window(
            col_off=border, row_off=border,
            width=src.width - 2 * border,
            height=src.height - 2 * border
        )
        data = src.read(window=win)
        transform = src.window_transform(win)
        profile = src.profile.copy()
        profile.update({
            "width": data.shape[2],
            "height": data.shape[1],
            "transform": transform
        })

    mem = MemoryFile()
    dst = mem.open(**profile)
    dst.write(data)
    memfiles.append(mem)
    src_files_to_mosaic.append(dst)

mosaic, out_trans = merge(src_files_to_mosaic, nodata=-999.9, method='min')
print(f"Mosaico: min={mosaic.min():.2f}, max={mosaic.max():.2f}")

output_mosaic = os.path.join(isostasia_folder, "mosaico_isostasia.tif")
with rasterio.open(
    output_mosaic, 'w', driver='GTiff',
    height=mosaic.shape[1], width=mosaic.shape[2],
    count=1, dtype=mosaic.dtype,
    crs=src_files_to_mosaic[0].crs,
    transform=out_trans, nodata=np.nan
) as dst:
    dst.write(mosaic[0], 1)

print(f"Mosaico guardado en: {output_mosaic}")